# Day 9 — What an expression really is

*100 Days of Polars · Phase 2: Expressions*

- **Post:** [What a Polars Expression Really Is](https://medium.com/python-in-plain-english/day-9-what-a-polars-expression-really-is-7be8d94afa67)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


An **expression** is a recipe for a column: *take `price`, multiply by 1.1*. It holds no data and runs nothing.
A context (`select`, `with_columns`, `filter`, `group_by().agg()`) hands it data and runs it.

In [2]:
items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
items.head(3)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""00010242fe8c5a6d1ba2dd792cb162…",1,"""4244733e06e7ecb4970a6e2683c13e…","""48436dade18ac8b2bce089ec2a0412…",2017-09-19 09:45:35,58.9,13.29
"""00018f77f2f0320c557190d7a144bd…",1,"""e5f2d52b802189ee658865ca93d83a…","""dd7ddc04e1b6c2c614352b383efe2d…",2017-05-03 11:05:13,239.9,19.93
"""000229ec398224ef6ca0657da4fc70…",1,"""c777355d18b72b67abbeef9df44fd0…","""5b51032eddd242adc84c38acab88f2…",2018-01-18 14:48:30,199.0,17.87


## 1. An expression is just a Python object

In [3]:
e = pl.col("price") * 1.1

print(type(e))
e  # the repr shows the recipe, not data

<class 'polars.expr.expr.Expr'>


<Expr ['[(col("price")) * (dyn float: …'] at 0x260DFD59F40>

In [4]:
# Compare: indexing a DataFrame gives a Series, computed right now
s = items["price"] * 1.1
print(type(s), s.len())

<class 'polars.series.series.Series'> 112650


## 2. One expression, four contexts

In [5]:
items.select(e).head(3)

price
f64
64.79
263.89
218.9


In [6]:
items.with_columns(price_with_tax=e).select("price", "price_with_tax").head(3)

price,price_with_tax
f64,f64
58.9,64.79
239.9,263.89
199.0,218.9


In [7]:
print(items.filter(e > 1_000).height, "items cost more than 1,000 BRL after tax")

1027 items cost more than 1,000 BRL after tax


In [8]:
items.group_by("seller_id").agg(e.sum()).sort("price", descending=True).head(3)

seller_id,price
str,f64
"""4869f7a5dfa277a7dca6462dcf3b52…",252419.893
"""53243585a1d6dc2643021fd1853d89…",245053.655
"""4a3ca9315b744ce9f8e93743614938…",220520.212


Same object, four different results. The expression says **what**; the context says **where** and **how many rows come out**.

## 3. `pl.lit`: a constant as an expression

In [9]:
items.select(
    "price",
    pl.lit(1.1).alias("tax_rate"),
    pl.lit("BRL").alias("currency"),
).head(3)  # a scalar literal is broadcast to every row

price,tax_rate,currency
f64,f64,str
58.9,1.1,"""BRL"""
239.9,1.1,"""BRL"""
199.0,1.1,"""BRL"""


In [10]:
# Gotcha: in select / with_columns a bare string is a COLUMN NAME, not a value
try:
    items.with_columns(currency="BRL")
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

ColumnNotFoundError -> unable to find column "BRL"; valid columns: ["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"]


In [11]:
items.with_columns(currency=pl.lit("BRL")).select("price", "currency").head(3)

price,currency
f64,str
58.9,"""BRL"""
239.9,"""BRL"""
199.0,"""BRL"""


## 4. Expressions compose like values

In [12]:
revenue = pl.col("price") + pl.col("freight_value")

items.select(
    total=revenue.sum(),
    avg=revenue.mean(),
    biggest=revenue.max(),
)

total,avg,biggest
f64,f64,f64
1.5844e7,140.644059,6929.31


In [13]:
# Build on an existing expression: nothing is computed until a context runs it
freight_share = pl.col("freight_value") / revenue

items.select(freight_share_pct=(freight_share * 100).round(1)).describe()

statistic,freight_share_pct
str,f64
"""count""",112650.0
"""null_count""",0.0
"""mean""",21.336306
"""std""",12.949522
"""min""",0.0
"""25%""",11.8
"""50%""",18.8
"""75%""",28.2
"""max""",96.3


## 5. Look inside an expression

In [14]:
print(freight_share.meta.root_names())   # which columns does it read?
print(freight_share.meta.output_name())  # what will the output column be called?
freight_share.meta.tree_format()         # the expression tree

['freight_value', 'price', 'freight_value']
freight_value
                0                   1                   2
   ┌────────────────────────────────────────────────────────────────
   │
   │      ╭───────────╮
 0 │      │ binary: / │
   │      ╰─────┬┬────╯
   │            ││
   │            │╰──────────────────────────────────────╮
   │            │                                       │
   │      ╭─────┴─────╮                       ╭─────────┴──────────╮
 1 │      │ binary: + │                       │ col(freight_value) │
   │      ╰─────┬┬────╯                       ╰────────────────────╯
   │            ││
   │            │╰──────────────────╮
   │            │                   │
   │  ╭─────────┴──────────╮  ╭─────┴──────╮
 2 │  │ col(freight_value) │  │ col(price) │
   │  ╰────────────────────╯  ╰────────────╯



In [ ]:
(pl.col("a") - pl.col("b")).meta.tree_format()

            0            1
   ┌───────────────────────────
   │
   │  ╭───────────╮
 0 │  │ binary: - │
   │  ╰─────┬┬────╯
   │        ││
   │        │╰───────────╮
   │        │            │
   │    ╭───┴────╮   ╭───┴────╮
 1 │    │ col(b) │   │ col(a) │
   │    ╰────────╯   ╰────────╯



The output name comes from the **leftmost** column in the expression. Keep that in mind: it explains a lot of `DuplicateError`s later.

## 6. Where the column name comes from

The leftmost column *as written* names the output. Aggregations keep that name.

In [15]:
r1 = pl.col("price") + pl.col("freight_value")
r2 = pl.col("freight_value") + pl.col("price")

print(items.select(r1.mean()).columns, items.select(r2.mean()).columns)  # same number, different names

['price'] ['freight_value']


In [16]:
# Two outputs with the same name collide
try:
    items.select(r1.mean(), pl.col("price").max())
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

DuplicateError -> projections contained duplicate output name 'price'. It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.


In [17]:
# Habit: name every computed output
print(items.select(avg_revenue=r1.mean()).shape)          # 1 row, 1 column
print(items.with_columns(avg_revenue=r1.mean()).shape)    # all columns + 1, every row
print(items.columns[-1])                                   # items itself is unchanged

(1, 1)
(112650, 8)
freight_value


## Try it yourself

1. Write one expression for "price per item including freight, rounded to 2 decimals" and use it in `select`, `with_columns` and `group_by("order_id").agg()`.
2. What does `(pl.lit(100) - pl.col("price")).meta.output_name()` return? Why?
3. Build `is_expensive = pl.col("price") > 500`. Use it in `filter`, then inside `agg` as `is_expensive.sum()` per seller.